# Anti-collapse benchmark — 4 техники против prediction collapse

TimeXer на наших данных страдает от prediction collapse: при ~851k
параметров и ~108k сэмплов модель сходится к константному выходу
≈ маргинальной P(UP). Этот ноутбук замеряет эффект четырёх известных
anti-collapse техник, **запуская их параллельно в одной таблице
сравнения** на одних и тех же предобработанных данных.

| вариант | техника | референс |
|---|---|---|
| `baseline` | без новых техник (контроль) | — |
| `r_drop` | R-Drop (двойной forward + симметричный KL между двумя дропаут-семплами) | arXiv:2106.14448 |
| `mixup` | Mixup (λ-интерполяция между сэмплами) | arXiv:1710.09412 |
| `droppath` | Stochastic Depth в encoder-блоках | arXiv:1603.09382 / DeiT |
| `mtl_uncertainty` | Kendall learnable log-σ per horizon | arXiv:1705.07115 |
| `combined` | все четыре вместе с приглушёнными коэффициентами | — |

**Почему sequential, а не multiprocessing:** на одиночной GPU в Colab
параллельный запуск 5+ моделей конкурирует за compute и не даёт
ускорения. Sequential на L4: ~5 мин на вариант, итого ~25-30 мин.

Метрики на выходе для каждого варианта:
- `best_val_loss` (best epoch), `final_train_loss`, `final_val_loss`
- `prob_max` / `prob_mean` / `prob_std` (test) — индикатор collapse
- `bayes_T` / `bayes_n_buy` / `bayes_sharpe` / `bayes_total_return`

В summary-таблице сразу видно: какая техника подняла `prob_max` выше
0.5 (декомпрессировала выход) и какая принесла реальный edge на
бэктесте.


## 0. Окружение

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        print(f'{PROJECT_ROOT} существует — pull...')
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH],
                       check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard',
                        f'origin/{REPO_BRANCH}'], check=True, env=env)
        head = subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'rev-parse', '--short', 'HEAD'],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        print(f'HEAD={head}')
    else:
        subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
             f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
            check=True, env=env,
        )
        print(f'Clone -> {PROJECT_ROOT}')
else:
    PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}, IN_COLAB: {IN_COLAB}')


In [ ]:
if IN_COLAB:
    print('Устанавливаем зависимости...')
    subprocess.run(
        ['pip', 'install', '--quiet',
         'torch>=2.2', 'pandas>=2.1', 'pyarrow>=15.0',
         'scikit-learn>=1.4', 'requests>=2.31', 'yfinance>=0.2.40',
         'pydantic>=2.6', 'tqdm>=4.66', 'ipywidgets>=8.0'],
        check=True,
    )
    print('Готово.')


In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if not (SRC / 'graduate_work' / '__init__.py').exists():
    raise FileNotFoundError(f'graduate_work не найден в {SRC}')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import dataclasses
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from graduate_work.config import default_config

cfg = default_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Тикеры:', cfg.data.tickers)
print('Горизонты:', cfg.data.horizons)
print('Окно:', cfg.data.window_size)
print('Архитектура:', cfg.model.architecture)
print('Device:', device)


## 1. Drive + датасет (один раз для всех вариантов)

In [ ]:
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    drive_raw = DRIVE_BASE / 'data' / 'raw'
    if drive_raw.exists():
        shutil.copytree(drive_raw, cfg.paths.data_raw, dirs_exist_ok=True)
        n = sum(1 for _ in cfg.paths.data_raw.rglob('*.csv'))
        print(f'Подтянули {n} CSV')
    else:
        raise FileNotFoundError(f'Нет {drive_raw} - сначала запусти training_pipeline.ipynb')


In [ ]:
from graduate_work.features import build_dataset

prepared = build_dataset(cfg.data, cfg.paths, persist=True, trading_cfg=cfg.trading)
print(f'train: {prepared.train["x"].shape}')
print(f'  val: {prepared.val["x"].shape}')
print(f' test: {prepared.test["x"].shape}')
print()
for split_name, split in (('train', prepared.train),
                          ('  val', prepared.val),
                          (' test', prepared.test)):
    line = f'  {split_name}:'
    for j, hz in enumerate(cfg.data.horizons):
        line += f'  h={hz:>2d}={float((split["y"][:, j] >= 0.5).mean()):.3f}'
    print(line)


## 2. Helpers — реализации anti-collapse техник

Все хелперы — локально в ноутбуке, чтобы не трогать пакетный код.
Каждая техника изолирована и активируется флагом в variant config.


In [ ]:
# --- DropPath (Stochastic Depth) ---
class DropPath(nn.Module):
    """Per-sample drop residual branch with prob `drop_prob`.

    arXiv:1603.09382. Разделяется по batch dim:
    `mask ~ Bernoulli(1 - drop_prob)`, output = x / (1 - drop_prob) * mask.
    """
    def __init__(self, drop_prob: float = 0.0) -> None:
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep)
        return x.div(keep) * mask


class _DropPathEncoderLayer(nn.Module):
    """Wrapper над TimeXer's _EncoderLayer с DropPath на residuals.

    Сохраняет интерфейс родителя, но внутри:
        x = norm1(x + drop_path(self_attn(x)))
        x = norm2(x + drop_path(ffn(x)))
    """
    def __init__(self, base: nn.Module, drop_prob: float) -> None:
        super().__init__()
        self.base = base
        self.drop_path = DropPath(drop_prob)

    def forward(self, x: torch.Tensor, exo: torch.Tensor | None = None) -> torch.Tensor:
        out, _ = self.base.self_attn(x, x, x)
        x = self.base.norm1(x + self.drop_path(out))
        if self.base.cross is not None and exo is not None:
            x = self.base.cross(x, exo)
        x = self.base.norm2(x + self.drop_path(self.base.ffn(x)))
        return x


def apply_droppath(model: nn.Module, drop_path_max: float) -> nn.Module:
    """In-place wrap каждый encoder layer DropPath'ом, линейный schedule 0..p_max."""
    if not hasattr(model, 'layers') or drop_path_max <= 0:
        return model
    n = len(model.layers)
    probs = torch.linspace(0.0, drop_path_max, n).tolist()
    model.layers = nn.ModuleList([
        _DropPathEncoderLayer(layer, p) for layer, p in zip(model.layers, probs, strict=True)
    ])
    return model


In [ ]:
# --- Mixup ---
def mixup_batch(x: torch.Tensor, y: torch.Tensor, alpha: float) -> tuple[torch.Tensor, torch.Tensor]:
    """λ ~ Beta(α, α); линейная интерполяция (x, y) с перестановкой батча.

    Применяется ТОЛЬКО на train, после feature-нормализации.
    """
    if alpha <= 0:
        return x, y
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.shape[0], device=x.device)
    return lam * x + (1.0 - lam) * x[perm], lam * y + (1.0 - lam) * y[perm]


# --- R-Drop symmetric KL ---
def symmetric_kl_bernoulli(logits_a: torch.Tensor, logits_b: torch.Tensor) -> torch.Tensor:
    """Симметричный KL между двумя Bernoulli-распределениями над логитами.

    Для multi-horizon (B, H) усредняет по batch и horizon.
    """
    p_a = torch.sigmoid(logits_a).clamp(1e-7, 1.0 - 1e-7)
    p_b = torch.sigmoid(logits_b).clamp(1e-7, 1.0 - 1e-7)
    kl_ab = p_a * (p_a.log() - p_b.log()) + (1 - p_a) * ((1 - p_a).log() - (1 - p_b).log())
    kl_ba = p_b * (p_b.log() - p_a.log()) + (1 - p_b) * ((1 - p_b).log() - (1 - p_a).log())
    return 0.5 * (kl_ab + kl_ba).mean()


In [ ]:
# --- Anti-collapse Trainer ---
from graduate_work.training import Trainer, set_seed
from graduate_work.training.losses import WeightedBCEWithLogits

class AntiCollapseTrainer(Trainer):
    """Расширение Trainer'а с опциональными hooks: R-Drop, Mixup, MTL-uncertainty.

    DropPath встроен В САМУ модель через apply_droppath перед `Trainer(...)`,
    поэтому здесь не нужен. Этот класс перехватывает только `_step` и
    добавляет per-horizon Kendall log-σ, если включено.
    """
    def __init__(
        self,
        *args,
        r_drop_alpha: float = 0.0,
        mixup_alpha: float = 0.0,
        mtl_uncertainty: bool = False,
        num_horizons: int = 4,
        **kwargs,
    ) -> None:
        super().__init__(*args, **kwargs)
        self.r_drop_alpha = float(r_drop_alpha)
        self.mixup_alpha = float(mixup_alpha)
        self.mtl_uncertainty = bool(mtl_uncertainty)
        if self.mtl_uncertainty:
            self.log_sigmas = nn.Parameter(torch.zeros(num_horizons, device=self.device))
            # Добавляем log_sigmas в группу оптимизатора.
            self.optimizer.add_param_group({'params': [self.log_sigmas]})

    def _per_horizon_bce(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """Per-horizon BCE с pos_weight (если выставлен в loss_fn)."""
        pw = getattr(self.loss_fn, 'pos_weight', None)
        per = F.binary_cross_entropy_with_logits(
            logits, target, reduction='none', pos_weight=pw,
        )
        return per.mean(dim=0)  # (H,)

    def _classification_loss(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """Скалярный loss с учётом MTL-uncertainty, если включено."""
        if self.mtl_uncertainty:
            per_h = self._per_horizon_bce(logits, target)        # (H,)
            precision = torch.exp(-self.log_sigmas)
            return (precision * per_h + self.log_sigmas).mean()
        return self.loss_fn(logits, target, None)

    def _step(self, x: torch.Tensor, y: torch.Tensor, *, train: bool) -> float:
        if not self._is_classification:
            return super()._step(x, y, train=train)
        x = x.to(self.device); y = y.to(self.device)
        if train and self.mixup_alpha > 0:
            x, y = mixup_batch(x, y, self.mixup_alpha)
        with torch.set_grad_enabled(train):
            preds_a = self.model(x)
            if train and self.r_drop_alpha > 0:
                preds_b = self.model(x)  # другой dropout-семпл
                loss_a = self._classification_loss(preds_a, y)
                loss_b = self._classification_loss(preds_b, y)
                base = 0.5 * (loss_a + loss_b)
                kl = symmetric_kl_bernoulli(preds_a, preds_b)
                loss = base + self.r_drop_alpha * kl
            else:
                loss = self._classification_loss(preds_a, y)
            if train:
                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if self.cfg.grad_clip > 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
                self.optimizer.step()
        return float(loss.item())


## 3. Helper-runner — train + diagnose

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest
from graduate_work.backtest.engine import prices_from_full_frame
from graduate_work.model import build_model
from graduate_work.strategy import (
    SignalGenerator, attach_lr_targets, build_predictions_frame, calibrate_bayes_threshold,
)
from graduate_work.training import mc_predict

# Общий test_prices (нужен один раз для всех вариантов).
full = prepared.full_frame.copy()
if not isinstance(full.index, pd.DatetimeIndex):
    full.index = pd.to_datetime(full.index, utc=True)
test_start = pd.to_datetime(min(prepared.test['timestamp']), utc=True)
buffer = cfg.data.bar_timedelta * max(cfg.data.horizons)
test_end = pd.to_datetime(max(prepared.test['timestamp']), utc=True) + buffer
test_prices = prices_from_full_frame(
    full.loc[(full.index >= test_start) & (full.index <= test_end)],
)
bars_per_year = cfg.data.bars_per_year
print(f'test_prices: {len(test_prices)} баров, '
      f'{test_prices["ticker"].nunique()} тикер(ов)')


In [ ]:
def run_variant(name: str, *, drop_path_max: float = 0.0,
                r_drop_alpha: float = 0.0, mixup_alpha: float = 0.0,
                mtl_uncertainty: bool = False) -> dict:
    """Запустить один вариант: build → fit → MC-test → Bayes signals → backtest."""
    print(f'\n{"="*70}\n>>> VARIANT: {name}\n{"="*70}')
    print(f'  drop_path_max={drop_path_max}  r_drop_alpha={r_drop_alpha}  '
          f'mixup_alpha={mixup_alpha}  mtl={mtl_uncertainty}')
    t0 = time.time()
    set_seed(cfg.training.seed)

    # 1) Build TimeXer + опциональный DropPath.
    model = build_model(
        input_dim=prepared.num_features,
        num_horizons=len(cfg.data.horizons),
        cfg=cfg.model,
    )
    if drop_path_max > 0:
        model = apply_droppath(model, drop_path_max)
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  параметров: {n_params:,} (обучаемых: {n_trainable:,})')

    # 2) Fit с anti-collapse hooks.
    trainer = AntiCollapseTrainer(
        model, cfg.training,
        data_cfg=cfg.data, trading_cfg=cfg.trading,
        r_drop_alpha=r_drop_alpha, mixup_alpha=mixup_alpha,
        mtl_uncertainty=mtl_uncertainty,
        num_horizons=len(cfg.data.horizons),
    )
    history = trainer.fit(prepared.train, prepared.val)
    n_ep = len(history.train_loss)
    best_ep = int(np.argmin(history.val_loss)) + 1 if n_ep else 0
    best_val = float(min(history.val_loss)) if n_ep else float('nan')
    final_train = float(history.train_loss[-1]) if n_ep else float('nan')
    final_val = float(history.val_loss[-1]) if n_ep else float('nan')
    print(f'  best_ep={best_ep}/{n_ep}  best_val={best_val:.5f}  '
          f'final_train={final_train:.5f}  final_val={final_val:.5f}')

    # 3) MC inference on test + val.
    is_cls = cfg.data.mode == 'classification'
    mean, std = mc_predict(
        model, prepared.test['x'],
        mc_passes=cfg.training.mc_passes, batch_size=cfg.training.batch_size,
        apply_sigmoid=is_cls,
    )
    val_mean, val_std = mc_predict(
        model, prepared.val['x'],
        mc_passes=cfg.training.mc_passes, batch_size=cfg.training.batch_size,
        apply_sigmoid=is_cls,
    )
    prob_max = float(mean.max()); prob_mean = float(mean.mean())
    prob_std_mean = float(std.mean())
    n_above_05 = int((mean > 0.5).sum()); frac_above_05 = n_above_05 / mean.size
    print(f'  MC test:  prob mean={prob_mean:.4f}  max={prob_max:.4f}  '
          f'std mean={prob_std_mean:.4f}  #prob>0.5={n_above_05} ({frac_above_05*100:.2f}%)')

    # 4) Bayes signals + backtest.
    long_pred = build_predictions_frame(
        prepared.test['timestamp'], prepared.test['ticker'],
        mean, std, cfg.data.horizons,
    )
    val_pred = build_predictions_frame(
        prepared.val['timestamp'], prepared.val['ticker'],
        val_mean, val_std, cfg.data.horizons,
    )
    cost = 2.0 * (cfg.trading.commission_rate + cfg.trading.slippage_rate)
    lr_targets = attach_lr_targets(prepared.full_frame, prepared.val, cfg.data.horizons)
    bayes_calib = calibrate_bayes_threshold(val_pred, lr_targets, cost_per_trade=cost)
    tc = dataclasses.replace(
        cfg.trading,
        probability_threshold=bayes_calib.min_expected_return,
        selection_correction='none',
    )
    signals = SignalGenerator(tc, mode=cfg.data.mode).generate(long_pred)
    n_buy = int((signals['action'] == 'BUY').sum())
    if n_buy > 0:
        bt = run_backtest(signals, test_prices, tc)
        m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    else:
        m = {'sharpe': 0.0, 'total_return': 0.0, 'final_equity': float(cfg.trading.initial_capital),
             'max_drawdown': 0.0, 'win_rate': 0.0, 'n_trades': 0}
    print(f'  Bayes T={bayes_calib.min_expected_return:.4f}  BUY={n_buy}  '
          f'sharpe={m["sharpe"]:.3f}  return={m["total_return"]*100:.2f}%')

    elapsed = time.time() - t0
    print(f'  elapsed: {elapsed:.0f}s')
    return {
        'name': name, 'n_params': n_params, 'n_trainable': n_trainable,
        'best_epoch': best_ep, 'best_val': best_val,
        'final_train': final_train, 'final_val': final_val,
        'prob_mean': prob_mean, 'prob_max': prob_max,
        'prob_std_mean': prob_std_mean, 'frac_prob_above_05': frac_above_05,
        'bayes_T': float(bayes_calib.min_expected_return), 'n_buy': n_buy,
        'sharpe': float(m['sharpe']), 'total_return': float(m['total_return']),
        'final_equity': float(m['final_equity']),
        'max_dd': float(m['max_drawdown']), 'win_rate': float(m['win_rate']),
        'n_trades': int(m['n_trades']), 'elapsed_sec': elapsed,
    }


## 4. Sequential прогон 6 вариантов

Каждый вариант — независимое обучение TimeXer на одних и тех же
данных. Общее время на L4 ≈ 25-30 минут (5-6 минут на вариант).
Прогресс-бар в каждой эпохе показывает текущий вариант.


In [ ]:
variants = [
    {'name': 'baseline'},
    {'name': 'r_drop',          'r_drop_alpha': 1.0},
    {'name': 'mixup',           'mixup_alpha': 0.2},
    {'name': 'droppath',        'drop_path_max': 0.15},
    {'name': 'mtl_uncertainty', 'mtl_uncertainty': True},
    {'name': 'combined',        'r_drop_alpha': 0.5, 'mixup_alpha': 0.2,
                                'drop_path_max': 0.10, 'mtl_uncertainty': True},
]

results: list[dict] = []
for v in variants:
    results.append(run_variant(**v))


## 5. Сравнение результатов

In [ ]:
summary = pd.DataFrame(results).set_index('name')
print('=== ANTI-COLLAPSE BENCHMARK ===')
key_cols = [
    'best_val', 'prob_mean', 'prob_max', 'prob_std_mean', 'frac_prob_above_05',
    'n_buy', 'sharpe', 'total_return', 'win_rate', 'n_trades',
]
print(summary[key_cols].to_string(float_format=lambda x: f'{x:.4f}'))

# Декомпрессия выхода (главный индикатор успеха anti-collapse).
print()
print('=== DELTA относительно baseline ===')
base = summary.loc['baseline']
deltas = summary[['prob_max', 'prob_std_mean', 'sharpe', 'total_return', 'n_buy']].subtract(base)
print(deltas.to_string(float_format=lambda x: f'{x:+.4f}'))


In [ ]:
# --- Визуализация ---
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
metrics_to_plot = [
    ('prob_max',      'max(prob)\nдекомпрессия выхода',   '#1f77b4'),
    ('prob_std_mean', 'mean(MC std)\nepistemic uncertainty', '#2ca02c'),
    ('sharpe',        'Sharpe',                              '#d62728'),
    ('total_return',  'Total return',                        '#9467bd'),
]
for ax, (col, title, color) in zip(axes, metrics_to_plot, strict=False):
    vals = summary[col]
    bars = ax.bar(vals.index, vals.values, color=color)
    # Подсветка baseline-уровня.
    base_val = summary.loc['baseline', col]
    ax.axhline(base_val, color='gray', linestyle=':', linewidth=1, label='baseline')
    ax.set_title(title); ax.set_xlabel('variant')
    ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, vals.values, strict=False):
        ax.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, val),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', fontsize=8)
fig.tight_layout(); plt.show()


## 6. Интерпретация

**Главный диагностический сигнал** — `prob_max` и `prob_std_mean`.
Если они существенно выросли по сравнению с `baseline`, техника
**физически вытащила модель из коллапса**. Если sharpe тоже вырос —
edge реальный, не шум.

**Ожидаемые паттерны:**
- `r_drop` обычно даёт лучший рост `prob_max` (KL-штраф напрямую
  заставляет модель коммититься).
- `mixup` сильно увеличивает разнообразие train-распределения, но
  может ухудшить `best_val` (val-loss оценивается на «чистом»
  распределении).
- `droppath` стабильно немного улучшает все метрики — это «free
  lunch» регуляризатор для трансформеров small-data.
- `mtl_uncertainty` важен когда у горизонтов разный signal-to-noise
  (короткие горизонты дают плотный сигнал, длинные — слабый); даёт
  адаптивные веса вместо равных.
- `combined` — лучший в среднем, но коэффициенты могут конфликтовать;
  если одна техника явно сильнее, лучше оставить её одну с полным
  коэффициентом, чем размывать.

**Что делать дальше:**
- Лучшую технику — закрепляем как дефолт в `cfg.training`/`cfg.model`.
- Если `prob_max` нигде не превысил 0.55 — проблема не в архитектуре,
  а в данных (фичи не несут сигнала о направлении движения), и нужны
  внешние источники (см. ноутбук про consensus + short-модель).
